# 2일차 — 기준일 2026-08-31 미결 인보이스 에이징 (DPD 구간)

- 데이터: DuckDB `workbench/day2/data/d2_duckdb.db`의 `d1_invoices` (바이어 이름만 `d1_buyers_raw`에서). DB가 없으면 먼저 `uv run python scripts/d2_duckdb_load.py`.
- 정의 (요청 그대로)

| 항목 | 정의 | SQL |
|---|---|---|
| 기준일 | 2026-08-31 | `SET VARIABLE ref_date` → `getvariable('ref_date')` |
| 대상 | 발행일이 기준일 이전(**당일 포함**) | `invoice_date <= ref_date` |
| 미결 | 결제일이 없거나 기준일 뒤 | `settled_date IS NULL OR settled_date > ref_date` |
| DPD | 기준일 − 만기일 (일) | `date_diff('day', due_date, ref_date)` |
| 구간 | 정상(DPD ≤ 0) · 1-30 · 31-60 · 61-90 · 91+ | 구간표 `aging_buckets` |
| 금액 | `amount_usd` — 인보이스 전액, 발행일 환율 USD 환산 | |

- `status` 열은 **2026-09-30 기준** 상태라 미결 판정에 쓰지 않는다 (기준일 뒤 정보가 섞인다).
- 해석이 갈릴 수 있는 3가지(대손·부도 건, 선수금 부분, 기준일 당일 발행)는 정의를 바꾸지 않고 6번 셀에서 크기만 보인다.
- 결과 저장: `workbench/day2/outputs/d2_aging_by_buyer.csv` · `d2_aging_bucket_counts.csv`
- 커널은 `.venv`(Python 3.13). 마무리는 커널 재시작 후 전체 실행.
- VS Code DuckDB 확장이 같은 DB 파일을 연결해 두면 Windows에서는 파이썬이 그 파일을 열 수 없다. 그때는 1번 셀이 같은 CSV를 메모리로 직접 읽고 `데이터: CSV 직접`이라고 찍는다 (결과는 같다).

In [1]:
# 1) 설정 — 경로는 저장소 맨 위 폴더 기준, DB는 읽기 전용으로 연다
import os
from pathlib import Path

# VS Code(notebookFileRoot)든 명령줄 nbconvert(노트북 폴더에서 시작)든 같은 곳으로 맞춘다
os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))

import duckdb
import pandas as pd

DB = "workbench/day2/data/d2_duckdb.db"
REF_DATE = "2026-08-31"  # 기준일 — 바꾸면 아래 SQL이 모두 따라간다
OUT = Path("workbench/day2/outputs")

pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

# DB 파일을 못 열면(VS Code DuckDB 확장이 잡고 있을 때 등) 같은 CSV를 같은 방식(read_csv 자동 감지)으로 메모리에 올린다
# — scripts/d2_duckdb_load.py가 DB를 만드는 방식과 같아서 결과도 같다
CSV = {
    "d1_invoices": "workbench/day2/data/d1_invoices.csv",
    "d1_buyers_raw": "workbench/day2/data/d1_buyers_raw.csv",
}
try:
    con = duckdb.connect(DB, read_only=True)
    SOURCE = DB
except duckdb.IOException:
    con = duckdb.connect()  # 메모리 DB
    for t, path in CSV.items():
        con.execute(f"CREATE VIEW {t} AS SELECT * FROM read_csv('{path}', header = true)")
    SOURCE = "CSV 직접 — DB 파일이 다른 프로그램에 잡혀 있음"
print("데이터:", SOURCE)

con.execute(f"SET VARIABLE ref_date = DATE '{REF_DATE}'")
con.sql("SELECT getvariable('ref_date') AS 기준일, (SELECT count(*) FROM d1_invoices) AS 전체_인보이스").df()

데이터: CSV 직접 — DB 파일이 다른 프로그램에 잡혀 있음


,기준일,전체_인보이스
0,2026-08-31,5033


## SQL ① 구간표 + 기준 뷰 `aging_base` (미결 인보이스 · DPD · 구간)

- 구간을 표로 두면 경계를 바꿀 때 이 표만 고치면 되고, 건수가 0인 구간도 결과에 남는다. `NULL` = 그쪽 끝이 열려 있음.
- 뒤의 SQL ② · ③은 모두 이 뷰를 읽는다 — 미결 정의가 한 곳에만 있다.

In [2]:
# 2) 구간표 + 기준 뷰 (TEMP = 이 연결에서만 보이고 DB 파일에는 저장되지 않는다)
con.sql("""
CREATE OR REPLACE TEMP VIEW aging_buckets AS
SELECT * FROM (VALUES
    (1, '정상',  NULL, 0),      -- DPD <= 0 : 만기 전 또는 만기 당일
    (2, '1-30',  1,    30),
    (3, '31-60', 31,   60),
    (4, '61-90', 61,   90),
    (5, '91+',   91,   NULL)
) AS t(bucket_no, bucket, dpd_from, dpd_to)
""")

con.sql("""
CREATE OR REPLACE TEMP VIEW aging_base AS
WITH invoices_at_ref AS (            -- 대상: 발행일이 기준일 이전(당일 포함)
    SELECT *
    FROM d1_invoices
    WHERE invoice_date <= getvariable('ref_date')
),
open_at_ref AS (                     -- 미결: 결제일이 없거나 기준일 뒤
    SELECT *,
           date_diff('day', due_date, getvariable('ref_date')) AS dpd   -- DPD = 기준일 - 만기일
    FROM invoices_at_ref
    WHERE settled_date IS NULL
       OR settled_date > getvariable('ref_date')
)
SELECT o.invoice_id, o.buyer_id, o.country_code, o.payment_method,
       o.invoice_date, o.due_date, o.settled_date, o.status,
       o.amount_usd, o.advance_pct, o.dpd,
       b.bucket_no, b.bucket
FROM open_at_ref AS o
JOIN aging_buckets AS b
  ON (b.dpd_from IS NULL OR o.dpd >= b.dpd_from)
 AND (b.dpd_to   IS NULL OR o.dpd <= b.dpd_to)
""")

con.sql("""
SELECT count(*)                    AS "미결 건수",
       count(DISTINCT buyer_id)    AS "바이어 수",
       round(sum(amount_usd), 2)   AS "미결 금액(USD)",
       min(dpd)                    AS "최소 DPD",
       max(dpd)                    AS "최대 DPD"
FROM aging_base
""").df()

,미결 건수,바이어 수,미결 금액(USD),최소 DPD,최대 DPD
0,315,102,"4,793,470.12",-90,852


In [3]:
# 기준 뷰 앞 10행 (DPD가 큰 순)
con.sql("SELECT * FROM aging_base ORDER BY dpd DESC, invoice_id LIMIT 10").df()

,invoice_id,buyer_id,country_code,payment_method,invoice_date,due_date,settled_date,status,amount_usd,advance_pct,dpd,bucket_no,bucket
0,INV-2404-0003,B103,SAU,OA,2024-04-01,2024-05-01,NaT,written_off,"16,130.00",0.00,852,5,91+
1,INV-2404-0027,B099,TUR,DA,2024-04-07,2024-06-06,NaT,written_off,"13,260.00",0.00,816,5,91+
2,INV-2404-0063,B099,TUR,DA,2024-04-17,2024-06-16,NaT,written_off,"3,170.00",0.00,806,5,91+
3,INV-2404-0071,B099,TUR,DA,2024-04-19,2024-06-18,NaT,written_off,"27,990.00",0.00,804,5,91+
4,INV-2405-0033,B099,TUR,DA,2024-05-09,2024-07-08,NaT,written_off,"38,400.00",0.00,784,5,91+
5,INV-2405-0058,B099,TUR,DA,2024-05-14,2024-07-13,NaT,written_off,"5,120.00",0.00,779,5,91+
6,INV-2405-0101,B099,TUR,DA,2024-05-23,2024-07-22,NaT,written_off,"35,640.00",0.00,770,5,91+
7,INV-2405-0112,B099,TUR,DA,2024-05-25,2024-07-24,NaT,written_off,"4,960.00",0.00,768,5,91+
8,INV-2406-0001,B099,TUR,DA,2024-06-01,2024-07-31,NaT,written_off,"12,430.00",0.00,761,5,91+
9,INV-2406-0058,B099,TUR,DA,2024-06-14,2024-08-13,NaT,written_off,"9,000.00",0.00,748,5,91+


## SQL ② 바이어별 미결 USD 금액 × 구간

- `d1_buyers_raw`는 buyer_id가 3개 중복(B047 · B094 · B114)이라, 이름을 붙이기 전에 바이어당 1행으로 줄인다 — 그대로 조인하면 그 바이어의 금액이 두 배가 된다.
- 해당 구간에 미결이 없으면 0. 정렬은 합계가 큰 순.

In [4]:
# 3) 바이어별 구간 금액
SQL_BY_BUYER = """
WITH buyer_names AS (                -- buyer_id 중복 제거: 바이어당 1행
    SELECT buyer_id, min(buyer_name) AS buyer_name
    FROM d1_buyers_raw
    GROUP BY buyer_id
)
SELECT a.buyer_id,
       n.buyer_name,
       round(coalesce(sum(a.amount_usd) FILTER (WHERE a.bucket = '정상'),  0), 2) AS "정상",
       round(coalesce(sum(a.amount_usd) FILTER (WHERE a.bucket = '1-30'),  0), 2) AS "1-30",
       round(coalesce(sum(a.amount_usd) FILTER (WHERE a.bucket = '31-60'), 0), 2) AS "31-60",
       round(coalesce(sum(a.amount_usd) FILTER (WHERE a.bucket = '61-90'), 0), 2) AS "61-90",
       round(coalesce(sum(a.amount_usd) FILTER (WHERE a.bucket = '91+'),   0), 2) AS "91+",
       round(sum(a.amount_usd), 2)  AS "합계",
       count(*)                     AS "미결 건수",
       max(a.dpd)                   AS "최대 DPD"
FROM aging_base AS a
LEFT JOIN buyer_names AS n USING (buyer_id)
GROUP BY a.buyer_id, n.buyer_name
ORDER BY "합계" DESC, a.buyer_id
"""
by_buyer = con.sql(SQL_BY_BUYER).df()
print(f"미결이 있는 바이어 {len(by_buyer)}곳")
by_buyer

미결이 있는 바이어 102곳


,buyer_id,buyer_name,정상,1-30,31-60,61-90,91+,합계,미결 건수,최대 DPD
0,B034,Helvara Humanitarian Supply Agency,"231,887.48",0.00,0.00,0.00,0.00,"231,887.48",2,-9
1,B028,"Davura Kogyo Co., Ltd.","202,099.47",0.00,0.00,0.00,0.00,"202,099.47",11,-8
2,B005,"Xanolin Distribution Co., Ltd.","173,020.00","26,000.00",0.00,0.00,0.00,"199,020.00",11,29
3,B099,Helura Endüstriyel Ölçüm Ltd. Şti.,0.00,0.00,0.00,0.00,"190,740.00","190,740.00",12,816
4,B108,Nelella Distribution JSC,"175,980.00",0.00,0.00,0.00,0.00,"175,980.00",9,-14
5,B068,"Belion Automation (Thailand) Co., Ltd.","109,420.00","59,160.00",0.00,0.00,0.00,"168,580.00",7,10
6,B086,Oroasta Handelsgesellschaft AG,"165,153.31",0.00,0.00,0.00,0.00,"165,153.31",11,-10
7,B098,Zenyn Outlets Ltd.,"164,660.00",0.00,0.00,0.00,0.00,"164,660.00",3,-3
8,B057,"Calant Projects Co., Ltd.","138,790.00","2,400.00",0.00,0.00,0.00,"141,190.00",9,4
9,B148,Rovella Water Services FZE,"116,170.00",0.00,0.00,0.00,0.00,"116,170.00",3,-19


## SQL ③ 전체 구간별 건수 (+ 금액 · 비중)

- 구간표에서 출발해 LEFT JOIN 하므로 0건 구간도 한 줄로 나온다. 맨 아래 `합계` 줄.
- 금액 비중(%) = 구간 금액 ÷ 전체 미결 금액 × 100.

In [5]:
# 4) 전체 구간별 건수
SQL_BY_BUCKET = """
SELECT "구간", "건수", "바이어 수", "금액(USD)", "금액 비중(%)"
FROM (
    SELECT k.bucket_no                         AS ord,
           k.bucket                            AS "구간",
           count(a.invoice_id)                 AS "건수",
           count(DISTINCT a.buyer_id)          AS "바이어 수",
           round(coalesce(sum(a.amount_usd), 0), 2) AS "금액(USD)",
           round(100 * coalesce(sum(a.amount_usd), 0)
                 / (SELECT sum(amount_usd) FROM aging_base), 1) AS "금액 비중(%)"
    FROM aging_buckets AS k
    LEFT JOIN aging_base AS a USING (bucket_no)
    GROUP BY k.bucket_no, k.bucket
    UNION ALL
    SELECT 99, '합계', count(*), count(DISTINCT buyer_id), round(sum(amount_usd), 2), 100.0
    FROM aging_base
)
ORDER BY ord
"""
by_bucket = con.sql(SQL_BY_BUCKET).df()
by_bucket

,구간,건수,바이어 수,금액(USD),금액 비중(%)
0,정상,260,91,"3,997,837.21",83.40
1,1-30,21,18,"291,142.77",6.10
2,31-60,9,7,"164,380.14",3.40
3,61-90,1,1,"11,870.00",0.20
4,91+,24,7,"328,240.00",6.80
5,합계,315,102,"4,793,470.12",100.00


## 5) 검증 — 숫자가 서로 맞는지, 정의가 그대로 들어갔는지

하나라도 실패하면 `AssertionError`로 멈춘다.

In [6]:
# 5) 검증
base = con.sql("SELECT * FROM aging_base").df()
ref = pd.Timestamp(REF_DATE)
n_open_direct = con.sql("""
    SELECT count(*) FROM d1_invoices
    WHERE invoice_date <= getvariable('ref_date')
      AND (settled_date IS NULL OR settled_date > getvariable('ref_date'))
""").fetchone()[0]

checks = {
    "구간 JOIN 뒤에도 행이 줄거나 늘지 않음 (기준 뷰 = 정의 직접 계산)": len(base) == n_open_direct,
    "기준 뷰 invoice_id 중복 없음": base.invoice_id.is_unique,
    "기준일 뒤 발행 0건": int((base.invoice_date > ref).sum()) == 0,
    "기준일 이전 완납 0건": int((base.settled_date <= ref).sum()) == 0,
    "DPD = 기준일 - 만기일": bool(((ref - base.due_date).dt.days == base.dpd).all()),
    "바이어 표 buyer_id 중복 없음": by_buyer.buyer_id.is_unique,
    "바이어 표 합계 = 기준 뷰 금액": round(by_buyer["합계"].sum(), 2) == round(base.amount_usd.sum(), 2),
    "바이어 표 구간 5칸 합 = 바이어 합계": bool((by_buyer[["정상", "1-30", "31-60", "61-90", "91+"]].sum(axis=1).round(2) == by_buyer["합계"].round(2)).all()),
    "바이어 표 건수 합 = 기준 뷰 건수": int(by_buyer["미결 건수"].sum()) == len(base),
    "구간 표 건수 합 = 기준 뷰 건수": int(by_bucket.loc[by_bucket["구간"] != "합계", "건수"].sum()) == len(base),
    "구간 표 금액 합 = 기준 뷰 금액": round(by_bucket.loc[by_bucket["구간"] != "합계", "금액(USD)"].sum(), 2) == round(base.amount_usd.sum(), 2),
}
for name, ok in checks.items():
    print("OK  " if ok else "FAIL", name)
assert all(checks.values()), "검증 실패 — 위 FAIL 줄을 본다"

# 구간 경계 눈으로 확인: 각 구간의 DPD 최소·최대가 구간표 안에 있어야 한다
con.sql("""
SELECT bucket AS "구간", min(dpd) AS "DPD 최소", max(dpd) AS "DPD 최대", count(*) AS "건수"
FROM aging_base GROUP BY bucket_no, bucket ORDER BY bucket_no
""").df()

OK   구간 JOIN 뒤에도 행이 줄거나 늘지 않음 (기준 뷰 = 정의 직접 계산)
OK   기준 뷰 invoice_id 중복 없음
OK   기준일 뒤 발행 0건
OK   기준일 이전 완납 0건
OK   DPD = 기준일 - 만기일
OK   바이어 표 buyer_id 중복 없음
OK   바이어 표 합계 = 기준 뷰 금액
OK   바이어 표 구간 5칸 합 = 바이어 합계
OK   바이어 표 건수 합 = 기준 뷰 건수
OK   구간 표 건수 합 = 기준 뷰 건수
OK   구간 표 금액 합 = 기준 뷰 금액


,구간,DPD 최소,DPD 최대,건수
0,정상,-90,0,260
1,1-30,1,29,21
2,31-60,31,60,9
3,61-90,76,76,1
4,91+,117,852,24


## 6) 해석이 갈릴 수 있는 3가지 — 정의는 그대로 두고 크기만 본다

| # | 무엇 | 지금 처리 | 다르게 하려면 |
|---|---|---|---|
| ① | 대손(`written_off`) · 부도(`default`) 건 | 결제일이 없으므로 **미결에 포함** | 대손 처리 날짜가 데이터에 없다 — `status`(09-30 기준)로 빼면 기준일 뒤 정보를 쓰게 된다. 뺄지는 사람이 정한다 |
| ② | 선수금 부분(`advance_pct` > 0: `TT_SPLIT_30_70` 30%, 일부 `OA` 10%) | 인보이스 **전액**(`amount_usd`) | SQL ①의 `o.amount_usd`를 `o.amount_usd * (1 - o.advance_pct) AS amount_usd`로 |
| ③ | 기준일 당일(2026-08-31) 발행 | '이전'을 **당일 포함**으로 봄 | SQL ①의 `<=`를 `<`로 |

In [7]:
# 6) 3가지의 크기 (건수 · USD)
con.sql("""
SELECT '① 대손·부도(status 09-30 기준) 건' AS "항목",
       count(*) FILTER (WHERE status IN ('written_off', 'default'))                          AS "건수",
       round(coalesce(sum(amount_usd) FILTER (WHERE status IN ('written_off', 'default')), 0), 2) AS "USD",
       string_agg(DISTINCT bucket, ', ') FILTER (WHERE status IN ('written_off', 'default'))  AS "들어간 구간"
FROM aging_base
UNION ALL
SELECT '② 선수금 부분(전액 - 선수금 차감액)',
       count(*) FILTER (WHERE advance_pct > 0),
       round(sum(amount_usd * advance_pct), 2),
       string_agg(DISTINCT bucket, ', ') FILTER (WHERE advance_pct > 0)
FROM aging_base
UNION ALL
SELECT '③ 기준일 당일 발행',
       count(*) FILTER (WHERE invoice_date = getvariable('ref_date')),
       round(coalesce(sum(amount_usd) FILTER (WHERE invoice_date = getvariable('ref_date')), 0), 2),
       string_agg(DISTINCT bucket, ', ') FILTER (WHERE invoice_date = getvariable('ref_date'))
FROM aging_base
""").df()

,항목,건수,USD,들어간 구간
0,① 대손·부도(status 09-30 기준) 건,26,"352,770.00","31-60, 91+, 1-30"
1,② 선수금 부분(전액 - 선수금 차감액),15,"69,089.78","정상, 1-30"
2,③ 기준일 당일 발행,5,"20,843.37",정상


In [8]:
# 참고: 미결 판정(기준일 08-31) × status(09-30 기준) — 'paid'는 09-01~09-30에 완납된 건이라 08-31에는 미결이 맞다
con.sql("""
SELECT status, bucket AS "구간", count(*) AS "건수", round(sum(amount_usd), 2) AS "USD"
FROM aging_base GROUP BY status, bucket_no, bucket ORDER BY status, bucket_no
""").df()

,status,구간,건수,USD
0,default,1-30,1,"15,810.00"
1,default,31-60,1,"8,720.00"
2,default,91+,9,"114,050.00"
3,open,정상,155,"2,190,026.14"
4,open,1-30,5,"96,328.03"
5,open,31-60,5,"56,560.14"
6,paid,정상,105,"1,807,811.07"
7,paid,1-30,15,"179,004.74"
8,paid,31-60,3,"99,100.00"
9,paid,61-90,1,"11,870.00"


## 7) 저장 · 연결 닫기

엑셀에서 한글이 깨지지 않게 `utf-8-sig`로 쓴다. 연결을 닫아야 `scripts/d2_duckdb_load.py`를 다시 돌릴 때 잠금 오류가 나지 않는다.

In [9]:
# 7) 저장
by_buyer.to_csv(OUT / "d2_aging_by_buyer.csv", index=False, encoding="utf-8-sig")
by_bucket.to_csv(OUT / "d2_aging_bucket_counts.csv", index=False, encoding="utf-8-sig")
con.close()
print("저장:", (OUT / "d2_aging_by_buyer.csv").as_posix(), "·", (OUT / "d2_aging_bucket_counts.csv").as_posix())

저장: workbench/day2/outputs/d2_aging_by_buyer.csv · workbench/day2/outputs/d2_aging_bucket_counts.csv
